CSE425 Project

**Goal:** Build a deep unsupervised model capable of generating novel music pieces across
multiple genres such as Classical, Jazz, Rock, Pop, and Electronic.

### Imports

In [1]:
# Install dependencies in notebook environment
%pip install -q numpy torch pretty_midi mido matplotlib tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import math
import os
import random

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pretty_midi
from tqdm.auto import tqdm

In [ ]:
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

### Preprocessing

Preprocessing Pipeline
1. Convert MIDI into piano-roll or token-based representation
2. Normalize timing resolution (e.g., 16 steps per bar)
3. Segment sequences into fixed-length windows

In [ ]:
# -----------------------------
# Groove dataset path discovery
# -----------------------------
CANDIDATE_DATA_ROOTS = [
    Path("data"),
    Path("../data"),
    Path("/kaggle/working/data"),
    Path("/kaggle/input"),
]

def discover_data_root(candidates):
    for c in candidates:
        if c.exists():
            return c.resolve()
    return candidates[0].resolve()

DATA_ROOT = discover_data_root(CANDIDATE_DATA_ROOTS)

def find_groove_dir(data_root: Path) -> Path:
    # Prefer explicit groove folder names
    explicit_names = ["groove", "groove-midi", "groove_midi"]
    for name in explicit_names:
        p = data_root / name
        if p.exists() and p.is_dir():
            return p.resolve()

    # Fallback: pick any subtree containing midi files and 'groove' in path
    midi_candidates = list(data_root.rglob("*.mid")) + list(data_root.rglob("*.midi"))
    for m in midi_candidates:
        if "groove" in str(m).lower():
            return m.parent.resolve()

    # Last fallback: data_root itself
    return data_root.resolve()

GROOVE_DIR = find_groove_dir(DATA_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("GROOVE_DIR:", GROOVE_DIR)

# --------------------------------
# Piano-roll preprocessing helpers
# --------------------------------
FS = 16
WINDOW_SIZE = 128
STRIDE = 64
MIN_ACTIVE_NOTES = 4

def list_midi_files(root: Path):
    return sorted(list(root.rglob("*.mid")) + list(root.rglob("*.midi")))

def normalize_piano_roll(piano_roll: np.ndarray) -> np.ndarray:
    if piano_roll.size == 0:
        return piano_roll.astype(np.float32)
    piano_roll = np.clip(piano_roll, 0.0, 1.0)
    return piano_roll.astype(np.float32)

def binarize_piano_roll(piano_roll: np.ndarray, threshold: float = 0.0) -> np.ndarray:
    return (piano_roll > threshold).astype(np.float32)

def midi_to_piano_roll(midi_path: Path, fs: int = FS) -> np.ndarray:
    midi = pretty_midi.PrettyMIDI(str(midi_path))
    roll = midi.get_piano_roll(fs=fs).T
    roll = normalize_piano_roll(roll / 127.0)
    return binarize_piano_roll(roll)

def segment_piano_roll(
    piano_roll: np.ndarray,
    window_size: int = WINDOW_SIZE,
    stride: int = STRIDE,
    min_active_notes: int = MIN_ACTIVE_NOTES,
) -> np.ndarray:
    if piano_roll.ndim != 2 or piano_roll.shape[1] != 128:
        raise ValueError("Expected piano_roll shape: [time, 128]")

    windows = []
    total_steps = piano_roll.shape[0]

    if total_steps < window_size:
        pad_len = window_size - total_steps
        padded = np.pad(piano_roll, ((0, pad_len), (0, 0)), mode="constant")
        if padded.sum() >= min_active_notes:
            windows.append(padded)
        return np.asarray(windows, dtype=np.float32)

    for start in range(0, total_steps - window_size + 1, stride):
        window = piano_roll[start : start + window_size]
        if window.sum() >= min_active_notes:
            windows.append(window)

    if not windows:
        return np.zeros((0, window_size, 128), dtype=np.float32)
    return np.asarray(windows, dtype=np.float32)

def build_piano_roll_windows(midi_dir: Path) -> np.ndarray:
    windows = []
    midi_files = list_midi_files(midi_dir)
    print(f"Found {len(midi_files)} MIDI files")
    for midi_file in tqdm(midi_files, desc="Preprocessing MIDI"):
        try:
            roll = midi_to_piano_roll(midi_file, fs=FS)
            chunks = segment_piano_roll(
                roll,
                window_size=WINDOW_SIZE,
                stride=STRIDE,
                min_active_notes=MIN_ACTIVE_NOTES,
            )
            if len(chunks):
                windows.append(chunks)
        except Exception:
            continue

    if not windows:
        return np.zeros((0, WINDOW_SIZE, 128), dtype=np.float32)
    return np.concatenate(windows, axis=0).astype(np.float32)

# -------------------
# Tokenizer for Task3
# -------------------
class SimplePitchTokenizer:
    def __init__(self):
        self.pad_token = 128
        self.bos_token = 129
        self.eos_token = 130
        self.rest_token = 131
        self.vocab_size = 132

    def piano_roll_to_tokens(self, piano_roll: np.ndarray) -> np.ndarray:
        tokens = []
        for frame in piano_roll:
            active = np.where(frame > 0)[0]
            if active.size == 0:
                tokens.append(self.rest_token)
            else:
                tokens.append(int(active.max()))
        return np.asarray(tokens, dtype=np.int64)

    def tokens_to_piano_roll(self, tokens: np.ndarray) -> np.ndarray:
        roll = np.zeros((len(tokens), 128), dtype=np.float32)
        for t, token in enumerate(tokens):
            if 0 <= int(token) <= 127:
                roll[t, int(token)] = 1.0
        return roll

    def add_special_tokens(self, tokens: np.ndarray) -> np.ndarray:
        return np.concatenate(
            [
                np.array([self.bos_token], dtype=np.int64),
                tokens,
                np.array([self.eos_token], dtype=np.int64),
            ]
        )

def segment_token_sequence(tokens: np.ndarray, window_size: int, stride: int) -> np.ndarray:
    if len(tokens) < window_size:
        padded = np.pad(tokens, (0, window_size - len(tokens)), constant_values=128)
        return padded[None, :].astype(np.int64)
    windows = []
    for start in range(0, len(tokens) - window_size + 1, stride):
        windows.append(tokens[start : start + window_size])
    return np.asarray(windows, dtype=np.int64)

### Task 1 - LSTM Autoencoder

In [ ]:
class LSTMAutoencoder(nn.Module):
	def __init__(
		self,
		input_dim: int = 128,
		hidden_dim: int = 256,
		latent_dim: int = 64,
		num_layers: int = 2,
		dropout: float = 0.2,
	) -> None:
		super().__init__()
		self.input_dim = input_dim
		self.hidden_dim = hidden_dim
		self.latent_dim = latent_dim
		self.num_layers = num_layers

		self.encoder = nn.LSTM(
			input_size=input_dim,
			hidden_size=hidden_dim,
			num_layers=num_layers,
			dropout=dropout if num_layers > 1 else 0.0,
			batch_first=True,
		)
		self.to_latent = nn.Linear(hidden_dim, latent_dim)
		self.latent_to_hidden = nn.Linear(latent_dim, hidden_dim * num_layers)
		self.decoder = nn.LSTM(
			input_size=input_dim,
			hidden_size=hidden_dim,
			num_layers=num_layers,
			dropout=dropout if num_layers > 1 else 0.0,
			batch_first=True,
		)
		self.output_proj = nn.Linear(hidden_dim, input_dim)

	def encode(self, x: torch.Tensor) -> torch.Tensor:
		_, (h_n, _) = self.encoder(x)
		return self.to_latent(h_n[-1])

	def decode(self, z: torch.Tensor, seq_len: int) -> torch.Tensor:
		batch_size = z.size(0)
		init_h = torch.tanh(self.latent_to_hidden(z)).view(self.num_layers, batch_size, self.hidden_dim)
		init_c = torch.zeros_like(init_h)
		decoder_in = torch.zeros(batch_size, seq_len, self.input_dim, device=z.device)
		decoded, _ = self.decoder(decoder_in, (init_h, init_c))
		return torch.sigmoid(self.output_proj(decoded))

	def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
		z = self.encode(x)
		recon = self.decode(z, seq_len=x.size(1))
		return recon, z


### Task 2 - Variational Autoencoder

In [ ]:
class MusicVAE(nn.Module):
	def __init__(
		self,
		input_dim: int = 128,
		hidden_dim: int = 256,
		latent_dim: int = 64,
		num_layers: int = 2,
		dropout: float = 0.2,
	) -> None:
		super().__init__()
		self.input_dim = input_dim
		self.hidden_dim = hidden_dim
		self.latent_dim = latent_dim
		self.num_layers = num_layers

		self.encoder = nn.LSTM(
			input_size=input_dim,
			hidden_size=hidden_dim,
			num_layers=num_layers,
			dropout=dropout if num_layers > 1 else 0.0,
			batch_first=True,
		)
		self.mu_head = nn.Linear(hidden_dim, latent_dim)
		self.logvar_head = nn.Linear(hidden_dim, latent_dim)

		self.latent_to_hidden = nn.Linear(latent_dim, hidden_dim * num_layers)
		self.decoder = nn.LSTM(
			input_size=input_dim,
			hidden_size=hidden_dim,
			num_layers=num_layers,
			dropout=dropout if num_layers > 1 else 0.0,
			batch_first=True,
		)
		self.output_proj = nn.Linear(hidden_dim, input_dim)

	def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
		_, (h_n, _) = self.encoder(x)
		h = h_n[-1]
		mu = self.mu_head(h)
		logvar = self.logvar_head(h)
		return mu, logvar

	def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
		std = torch.exp(0.5 * logvar)
		eps = torch.randn_like(std)
		return mu + eps * std

	def decode(self, z: torch.Tensor, seq_len: int) -> torch.Tensor:
		batch_size = z.size(0)
		init_h = torch.tanh(self.latent_to_hidden(z)).view(self.num_layers, batch_size, self.hidden_dim)
		init_c = torch.zeros_like(init_h)
		decoder_in = torch.zeros(batch_size, seq_len, self.input_dim, device=z.device)
		decoded, _ = self.decoder(decoder_in, (init_h, init_c))
		return torch.sigmoid(self.output_proj(decoded))

	def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
		mu, logvar = self.encode(x)
		z = self.reparameterize(mu, logvar)
		recon = self.decode(z, seq_len=x.size(1))
		return recon, mu, logvar


def vae_loss(
	recon_x: torch.Tensor,
	x: torch.Tensor,
	mu: torch.Tensor,
	logvar: torch.Tensor,
	beta: float = 0.1,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
	recon = F.binary_cross_entropy(recon_x, x)
	kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
	total = recon + beta * kl
	return total, recon, kl

### Task 3 - Transformer

In [ ]:
class CausalTransformer(nn.Module):
	def __init__(
		self,
		vocab_size: int,
		d_model: int = 256,
		nhead: int = 8,
		num_layers: int = 4,
		dim_feedforward: int = 512,
		dropout: float = 0.1,
		max_seq_len: int = 512,
	) -> None:
		super().__init__()
		self.vocab_size = vocab_size
		self.max_seq_len = max_seq_len

		self.token_emb = nn.Embedding(vocab_size, d_model)
		self.pos_emb = nn.Embedding(max_seq_len, d_model)
		encoder_layer = nn.TransformerEncoderLayer(
			d_model=d_model,
			nhead=nhead,
			dim_feedforward=dim_feedforward,
			dropout=dropout,
			batch_first=True,
			activation="gelu",
		)
		self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
		self.norm = nn.LayerNorm(d_model)
		self.head = nn.Linear(d_model, vocab_size)

	def _causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
		return torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)

	def forward(self, tokens: torch.Tensor) -> torch.Tensor:
		bsz, seq_len = tokens.shape
		if seq_len > self.max_seq_len:
			raise ValueError(f"Input sequence length {seq_len} exceeds max_seq_len={self.max_seq_len}")

		pos = torch.arange(seq_len, device=tokens.device).unsqueeze(0).expand(bsz, -1)
		x = self.token_emb(tokens) + self.pos_emb(pos)
		mask = self._causal_mask(seq_len, tokens.device)
		x = self.encoder(x, mask=mask)
		x = self.norm(x)
		return self.head(x)

	@torch.no_grad()
	def generate(
		self,
		start_tokens: torch.Tensor,
		max_new_tokens: int,
		temperature: float = 1.0,
		top_k: int | None = 20,
	) -> torch.Tensor:
		self.eval()
		generated = start_tokens

		for _ in range(max_new_tokens):
			context = generated[:, -self.max_seq_len :]
			logits = self.forward(context)[:, -1, :]
			logits = logits / max(temperature, 1e-6)

			if top_k is not None:
				values, _ = torch.topk(logits, k=min(top_k, logits.shape[-1]))
				logits[logits < values[:, [-1]]] = -float("inf")

			probs = torch.softmax(logits, dim=-1)
			next_token = torch.multinomial(probs, num_samples=1)
			generated = torch.cat([generated, next_token], dim=1)

		return generated

### Evaluation

#### Metrics

In [ ]:
def reconstruction_mse(target: torch.Tensor, pred: torch.Tensor) -> torch.Tensor:
	return F.mse_loss(pred, target)


def kl_divergence(mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
	return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())


def token_perplexity(logits: torch.Tensor, targets: torch.Tensor, ignore_index: int | None = None) -> float:
	vocab_size = logits.shape[-1]
	flat_logits = logits.reshape(-1, vocab_size)
	flat_targets = targets.reshape(-1)
	ce = F.cross_entropy(flat_logits, flat_targets, ignore_index=ignore_index)
	return float(torch.exp(ce).item())


def save_loss_curve(values: list[float], output_path: Path, title: str, ylabel: str) -> None:
	output_path.parent.mkdir(parents=True, exist_ok=True)
	plt.figure(figsize=(8, 4))
	plt.plot(np.arange(1, len(values) + 1), values, marker="o", linewidth=1.5)
	plt.title(title)
	plt.xlabel("Epoch")
	plt.ylabel(ylabel)
	plt.grid(alpha=0.3)
	plt.tight_layout()
	plt.savefig(output_path)
	plt.close()



#### Pitch Histogram

In [ ]:
def pitch_histogram(piano_roll: np.ndarray) -> np.ndarray:
	if piano_roll.ndim != 2 or piano_roll.shape[1] != 128:
		raise ValueError("Expected piano_roll shape: [time, 128]")
	hist = piano_roll.sum(axis=0).astype(np.float64)
	total = hist.sum()
	if total == 0:
		return np.zeros(128, dtype=np.float64)
	return hist / total


def histogram_l1_distance(a: np.ndarray, b: np.ndarray) -> float:
	return float(np.abs(a - b).sum())


def compare_pitch_distributions(reference_roll: np.ndarray, generated_roll: np.ndarray) -> float:
	ref = pitch_histogram(reference_roll)
	gen = pitch_histogram(generated_roll)
	return histogram_l1_distance(ref, gen)

#### Rhythm Score

In [ ]:
def onset_density(piano_roll: np.ndarray) -> float:
	if piano_roll.ndim != 2 or piano_roll.shape[1] != 128:
		raise ValueError("Expected piano_roll shape: [time, 128]")
	activity = (piano_roll.sum(axis=1) > 0).astype(np.float32)
	return float(activity.mean())


def rhythm_similarity(reference_roll: np.ndarray, generated_roll: np.ndarray) -> float:
	ref_density = onset_density(reference_roll)
	gen_density = onset_density(generated_roll)
	return max(0.0, 1.0 - abs(ref_density - gen_density))

### End-to-End Kaggle Training and Generation
Run the cells below in order. They will:
1. Build/load processed windows from Groove MIDI files in the data folder
2. Train AE, VAE, and Transformer
3. Save plots/checkpoints
4. Generate MIDI samples for deliverables

In [ ]:
# -------------------------
# Data build/load and split
# -------------------------
PROCESSED_DIR = Path("outputs/processed")
CHECKPOINT_DIR = Path("outputs/checkpoints")
PLOT_DIR = Path("outputs/plots")
MIDI_OUT_DIR = Path("outputs/generated_midis")

for d in [PROCESSED_DIR, CHECKPOINT_DIR, PLOT_DIR, MIDI_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ROLL_WINDOWS_PATH = PROCESSED_DIR / "piano_roll_windows.npy"
TOKEN_WINDOWS_PATH = PROCESSED_DIR / "token_windows.npy"

if ROLL_WINDOWS_PATH.exists():
    roll_windows = np.load(ROLL_WINDOWS_PATH)
    print("Loaded cached roll windows:", roll_windows.shape)
else:
    roll_windows = build_piano_roll_windows(GROOVE_DIR)
    np.save(ROLL_WINDOWS_PATH, roll_windows)
    print("Built roll windows:", roll_windows.shape)

if len(roll_windows) == 0:
    raise RuntimeError("No usable MIDI windows found. Verify Groove files are inside the data folder.")

def split_roll_dataset(windows: np.ndarray, val_ratio: float = 0.1):
    tensor = torch.tensor(windows, dtype=torch.float32)
    dataset = TensorDataset(tensor)
    train_size = int((1.0 - val_ratio) * len(dataset))
    val_size = len(dataset) - train_size
    train_set, val_set = random_split(dataset, [train_size, val_size])
    return train_set, val_set

AE_BATCH_SIZE = 32
VAE_BATCH_SIZE = 32
TR_BATCH_SIZE = 32

ae_train_set, ae_val_set = split_roll_dataset(roll_windows, val_ratio=0.1)
ae_train_loader = DataLoader(ae_train_set, batch_size=AE_BATCH_SIZE, shuffle=True)
ae_val_loader = DataLoader(ae_val_set, batch_size=AE_BATCH_SIZE, shuffle=False)

vae_train_set, vae_val_set = split_roll_dataset(roll_windows, val_ratio=0.1)
vae_train_loader = DataLoader(vae_train_set, batch_size=VAE_BATCH_SIZE, shuffle=True)
vae_val_loader = DataLoader(vae_val_set, batch_size=VAE_BATCH_SIZE, shuffle=False)

TOKENIZER = SimplePitchTokenizer()
TOKEN_WINDOW_SIZE = 129
TOKEN_STRIDE = 64

if TOKEN_WINDOWS_PATH.exists():
    token_windows = np.load(TOKEN_WINDOWS_PATH)
    print("Loaded cached token windows:", token_windows.shape)
else:
    token_chunks = []
    for roll in roll_windows:
        tokens = TOKENIZER.piano_roll_to_tokens(roll)
        tokens = TOKENIZER.add_special_tokens(tokens)
        chunks = segment_token_sequence(tokens, window_size=TOKEN_WINDOW_SIZE, stride=TOKEN_STRIDE)
        if len(chunks):
            token_chunks.append(chunks)
    token_windows = np.concatenate(token_chunks, axis=0) if len(token_chunks) else np.zeros((0, TOKEN_WINDOW_SIZE), dtype=np.int64)
    np.save(TOKEN_WINDOWS_PATH, token_windows)
    print("Built token windows:", token_windows.shape)

if len(token_windows) == 0:
    raise RuntimeError("No token windows available for Transformer training.")

tr_x = torch.tensor(token_windows[:, :-1], dtype=torch.long)
tr_y = torch.tensor(token_windows[:, 1:], dtype=torch.long)
tr_dataset = TensorDataset(tr_x, tr_y)
tr_train_size = int(0.9 * len(tr_dataset))
tr_val_size = len(tr_dataset) - tr_train_size
tr_train_set, tr_val_set = random_split(tr_dataset, [tr_train_size, tr_val_size])
tr_train_loader = DataLoader(tr_train_set, batch_size=TR_BATCH_SIZE, shuffle=True)
tr_val_loader = DataLoader(tr_val_set, batch_size=TR_BATCH_SIZE, shuffle=False)

print("AE train/val:", len(ae_train_set), len(ae_val_set))
print("Transformer train/val:", len(tr_train_set), len(tr_val_set))

In [ ]:
# ----------------
# Training routines
# ----------------
def train_ae_model(train_loader, val_loader, epochs=20, lr=1e-3):
    model = LSTMAutoencoder().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses = [], []

    for epoch in range(epochs):
        model.train()
        running = 0.0
        for (x,) in tqdm(train_loader, desc=f"AE Epoch {epoch+1}/{epochs}"):
            x = x.to(DEVICE)
            recon, _ = model(x)
            loss = F.binary_cross_entropy(recon, x)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += loss.item() * x.size(0)
        train_loss = running / len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for (x,) in val_loader:
                x = x.to(DEVICE)
                recon, _ = model(x)
                val_running += F.binary_cross_entropy(recon, x).item() * x.size(0)
        val_loss = val_running / max(1, len(val_loader.dataset))
        val_losses.append(val_loss)
        print(f"AE {epoch+1}: train={train_loss:.4f}, val={val_loss:.4f}")

    torch.save({"model_state_dict": model.state_dict()}, CHECKPOINT_DIR / "ae.pt")
    save_loss_curve(train_losses, PLOT_DIR / "ae_train_loss.png", "AE Reconstruction Loss", "BCE Loss")
    save_loss_curve(val_losses, PLOT_DIR / "ae_val_loss.png", "AE Validation Loss", "BCE Loss")
    return model, train_losses, val_losses

def train_vae_model(train_loader, val_loader, epochs=25, lr=1e-3, beta=0.1):
    model = MusicVAE().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_losses, val_losses, train_kls = [], [], []

    for epoch in range(epochs):
        model.train()
        running = 0.0
        running_kl = 0.0
        for (x,) in tqdm(train_loader, desc=f"VAE Epoch {epoch+1}/{epochs}"):
            x = x.to(DEVICE)
            recon, mu, logvar = model(x)
            loss, _, kl = vae_loss(recon, x, mu, logvar, beta=beta)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += loss.item() * x.size(0)
            running_kl += kl.item() * x.size(0)

        train_loss = running / len(train_loader.dataset)
        train_kl = running_kl / len(train_loader.dataset)
        train_losses.append(train_loss)
        train_kls.append(train_kl)

        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for (x,) in val_loader:
                x = x.to(DEVICE)
                recon, mu, logvar = model(x)
                loss, _, _ = vae_loss(recon, x, mu, logvar, beta=beta)
                val_running += loss.item() * x.size(0)
        val_loss = val_running / max(1, len(val_loader.dataset))
        val_losses.append(val_loss)
        print(f"VAE {epoch+1}: train={train_loss:.4f}, kl={train_kl:.4f}, val={val_loss:.4f}")

    torch.save({"model_state_dict": model.state_dict()}, CHECKPOINT_DIR / "vae.pt")
    save_loss_curve(train_losses, PLOT_DIR / "vae_train_total_loss.png", "VAE Total Loss", "Loss")
    save_loss_curve(val_losses, PLOT_DIR / "vae_val_total_loss.png", "VAE Validation Loss", "Loss")
    save_loss_curve(train_kls, PLOT_DIR / "vae_train_kl_loss.png", "VAE KL Divergence", "KL")
    return model, train_losses, val_losses, train_kls

def train_transformer_model(train_loader, val_loader, vocab_size=132, epochs=20, lr=3e-4):
    model = CausalTransformer(
        vocab_size=vocab_size,
        d_model=256,
        nhead=8,
        num_layers=4,
        dim_feedforward=512,
        dropout=0.1,
        max_seq_len=TOKEN_WINDOW_SIZE,
    ).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    train_losses, val_losses, val_ppls = [], [], []

    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in tqdm(train_loader, desc=f"TR Epoch {epoch+1}/{epochs}"):
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = F.cross_entropy(
                logits.reshape(-1, vocab_size),
                yb.reshape(-1),
                ignore_index=TOKENIZER.pad_token,
            )
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)
        train_loss = running / len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                logits = model(xb)
                loss = F.cross_entropy(
                    logits.reshape(-1, vocab_size),
                    yb.reshape(-1),
                    ignore_index=TOKENIZER.pad_token,
                )
                val_running += loss.item() * xb.size(0)
        val_loss = val_running / max(1, len(val_loader.dataset))
        val_ppl = float(np.exp(val_loss))
        val_losses.append(val_loss)
        val_ppls.append(val_ppl)
        print(f"TR {epoch+1}: train={train_loss:.4f}, val={val_loss:.4f}, ppl={val_ppl:.2f}")

    torch.save({"model_state_dict": model.state_dict()}, CHECKPOINT_DIR / "transformer.pt")
    save_loss_curve(train_losses, PLOT_DIR / "transformer_train_loss.png", "Transformer Train CE Loss", "Loss")
    save_loss_curve(val_losses, PLOT_DIR / "transformer_val_loss.png", "Transformer Validation CE Loss", "Loss")
    save_loss_curve(val_ppls, PLOT_DIR / "transformer_val_perplexity.png", "Transformer Validation Perplexity", "Perplexity")
    return model, train_losses, val_losses, val_ppls

In [ ]:
# -------------------
# Train all 3 models
# -------------------
# Adjust epochs if needed for faster experimentation.
AE_EPOCHS = 20
VAE_EPOCHS = 25
TR_EPOCHS = 20

RUN_AE = True
RUN_VAE = True
RUN_TR = True

ae_model = None
vae_model = None
tr_model = None

if RUN_AE:
    ae_model, ae_train_losses, ae_val_losses = train_ae_model(
        ae_train_loader, ae_val_loader, epochs=AE_EPOCHS, lr=1e-3
    )

if RUN_VAE:
    vae_model, vae_train_losses, vae_val_losses, vae_train_kls = train_vae_model(
        vae_train_loader, vae_val_loader, epochs=VAE_EPOCHS, lr=1e-3, beta=0.1
    )

if RUN_TR:
    tr_model, tr_train_losses, tr_val_losses, tr_val_ppls = train_transformer_model(
        tr_train_loader, tr_val_loader, vocab_size=TOKENIZER.vocab_size, epochs=TR_EPOCHS, lr=3e-4
    )

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Plot directory:", PLOT_DIR)

In [ ]:
# -------------------------
# MIDI export and sampling
# -------------------------
def piano_roll_to_pretty_midi(piano_roll: np.ndarray, fs: int = FS, velocity: int = 90):
    if piano_roll.ndim != 2 or piano_roll.shape[1] != 128:
        raise ValueError("Expected piano_roll shape: [time, 128]")
    pm = pretty_midi.PrettyMIDI()
    instrument = pretty_midi.Instrument(program=0)
    min_dur = 1.0 / fs

    for pitch in range(128):
        active = np.where(piano_roll[:, pitch] > 0)[0]
        if active.size == 0:
            continue
        runs = np.split(active, np.where(np.diff(active) != 1)[0] + 1)
        for run in runs:
            start = float(run[0]) / fs
            end = float(run[-1] + 1) / fs
            if end - start < min_dur:
                end = start + min_dur
            instrument.notes.append(
                pretty_midi.Note(velocity=velocity, pitch=pitch, start=start, end=end)
            )

    pm.instruments.append(instrument)
    return pm

def tokens_to_pretty_midi(tokens: np.ndarray, tokenizer: SimplePitchTokenizer, fs: int = FS):
    clean = [t for t in tokens.tolist() if 0 <= int(t) <= 127 or int(t) == tokenizer.rest_token]
    roll = tokenizer.tokens_to_piano_roll(np.asarray(clean, dtype=np.int64))
    return piano_roll_to_pretty_midi(roll, fs=fs)

def save_midi(pm: pretty_midi.PrettyMIDI, output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    pm.write(str(output_path))

@torch.no_grad()
def generate_ae_samples(model: LSTMAutoencoder, n_samples: int = 5, seq_len: int = WINDOW_SIZE):
    model.eval()
    z = torch.randn(n_samples, 64, device=DEVICE)
    rolls = model.decode(z, seq_len=seq_len).cpu().numpy()
    rolls = (rolls > 0.5).astype(np.float32)
    for i, roll in enumerate(rolls, start=1):
        pm = piano_roll_to_pretty_midi(roll, fs=FS)
        save_midi(pm, MIDI_OUT_DIR / f"ae_sample_{i:02d}.mid")

@torch.no_grad()
def generate_vae_samples(model: MusicVAE, n_samples: int = 8, seq_len: int = WINDOW_SIZE):
    model.eval()
    z = torch.randn(n_samples, 64, device=DEVICE)
    rolls = model.decode(z, seq_len=seq_len).cpu().numpy()
    rolls = (rolls > 0.5).astype(np.float32)
    for i, roll in enumerate(rolls, start=1):
        pm = piano_roll_to_pretty_midi(roll, fs=FS)
        save_midi(pm, MIDI_OUT_DIR / f"vae_sample_{i:02d}.mid")

@torch.no_grad()
def generate_transformer_samples(model: CausalTransformer, tokenizer: SimplePitchTokenizer, n_samples: int = 10, seq_len: int = 256):
    model.eval()
    start = torch.full((n_samples, 1), tokenizer.bos_token, dtype=torch.long, device=DEVICE)
    generated = model.generate(start, max_new_tokens=seq_len, temperature=1.0)
    generated = generated[:, 1:].cpu().numpy()
    for i, seq in enumerate(generated, start=1):
        pm = tokens_to_pretty_midi(seq, tokenizer=tokenizer, fs=FS)
        save_midi(pm, MIDI_OUT_DIR / f"transformer_sample_{i:02d}.mid")

In [ ]:
# -------------------
# Generate deliverables
# -------------------
if ae_model is None and (CHECKPOINT_DIR / "ae.pt").exists():
    ae_model = LSTMAutoencoder().to(DEVICE)
    ae_model.load_state_dict(torch.load(CHECKPOINT_DIR / "ae.pt", map_location=DEVICE)["model_state_dict"])

if vae_model is None and (CHECKPOINT_DIR / "vae.pt").exists():
    vae_model = MusicVAE().to(DEVICE)
    vae_model.load_state_dict(torch.load(CHECKPOINT_DIR / "vae.pt", map_location=DEVICE)["model_state_dict"])

if tr_model is None and (CHECKPOINT_DIR / "transformer.pt").exists():
    tr_model = CausalTransformer(
        vocab_size=TOKENIZER.vocab_size,
        d_model=256,
        nhead=8,
        num_layers=4,
        dim_feedforward=512,
        dropout=0.1,
        max_seq_len=TOKEN_WINDOW_SIZE,
    ).to(DEVICE)
    tr_model.load_state_dict(torch.load(CHECKPOINT_DIR / "transformer.pt", map_location=DEVICE)["model_state_dict"])

if ae_model is not None:
    generate_ae_samples(ae_model, n_samples=5, seq_len=WINDOW_SIZE)
if vae_model is not None:
    generate_vae_samples(vae_model, n_samples=8, seq_len=WINDOW_SIZE)
if tr_model is not None:
    generate_transformer_samples(tr_model, tokenizer=TOKENIZER, n_samples=10, seq_len=256)

print("Generated MIDI files saved to:", MIDI_OUT_DIR)
print("Use Kaggle file browser to download outputs from this directory.")